# 07 Dynamically Load Adapters into vLLM

This notebook loads local PEFT adapters into a running vLLM server.

```mermaid
sequenceDiagram
    participant N as Notebook
    participant V as vLLM
    N->>V: POST /v1/load_lora_adapter
    N->>V: GET /v1/models
    V-->>N: base plus LoRA model names
```

In [ ]:
from pathlib import Path
import os
import sys

CURRENT_DIR = Path.cwd()
PROJECT_ROOT = CURRENT_DIR if (CURRENT_DIR / "PROJECT_SPEC.md").exists() else CURRENT_DIR.parent
NOTEBOOK_DIR = PROJECT_ROOT / "notebooks"
sys.path.append(str(PROJECT_ROOT))

from training.config import DEFAULT_CONFIG

cfg = DEFAULT_CONFIG.copy()
cfg["data_dir"] = str((NOTEBOOK_DIR / cfg["data_dir"]).resolve())
cfg["output_dir"] = str((NOTEBOOK_DIR / cfg["output_dir"]).resolve())
os.environ["DATA_DIR"] = cfg["data_dir"]
os.environ["ADAPTER_DIR"] = cfg["output_dir"]
os.environ.setdefault("MLFLOW_EXPERIMENT_NAME", cfg["experiment_name"])

from llmops_demo.settings import ensure_dirs, settings

settings_cfg = settings()
ensure_dirs(Path(cfg["data_dir"]), Path(cfg["output_dir"]))

print(f"Project root: {PROJECT_ROOT}")
print(f"Data dir: {cfg['data_dir']}")
print(f"Adapter output dir: {cfg['output_dir']}")
print(f"Base model: {settings_cfg.base_model}")
print(f"Adapters: {settings_cfg.adapters}")

## Load adapters

Expected output: one success line per adapter.

In [ ]:
from scripts.load_adapters import load_adapter

for adapter in settings_cfg.adapters:
    load_adapter(
        settings_cfg.vllm_base_url,
        settings_cfg.vllm_api_key,
        adapter,
        Path(cfg["output_dir"]) / adapter,
    )

## Verify vLLM model registration

Expected output: `finance`, `legal`, and `healthcare` appear in `/v1/models`.

In [ ]:
import requests

response = requests.get(
    f"{settings_cfg.vllm_base_url}/v1/models",
    headers={"Authorization": f"Bearer {settings_cfg.vllm_api_key}"},
    timeout=10,
)
response.raise_for_status()
print(response.json())